# பாடம் 08 - பன்முகப்பட்ட முகவர் வடிவமைப்பு முறை


## அமைப்பு


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ஏன் பன்முக முகவர் அமைப்புகள்?

பயண திட்டமிடல் போன்ற உண்மை உலக பணிகள் பல விதமான நிபுணத்துவங்களை requirement க்கின்றன — லாஜிஸ்டிக்ஸ், உள்ளூர் அறிவு, பட்ஜெட்டிங் மற்றும் பல. ஒற்றை முகவர் அனைத்தையும் கையாள முயன்றால் விரைவாக சிரமமாகிவிடும்.

பன்முக முகவர் அமைப்புகள் இதனை **துறைசார் நிபுணத்துவம்** மூலம் தீர்க்கின்றன: ஒவ்வொரு முகவரும் ஒரு நிபுணத்துவ பகுதிக்கு கவனம் செலுத்தி, ஒரு பொது நபரைவிட அதிகத் தரமான முடிவுகளை வழங்குகின்றன. அவை **பரிமாணவாதத்தையும்** மேம்படுத்துகின்றன — புதிய முகவர்களை (எ.கா., விமான நிபுணர், உணவகம் விமர்சகர்) தற்போதைய பணிமுறை மறுதழுவாமல் சேர்க்கலாம். முகவர்கள் ஒருவரிடமிருந்து மற்றவருக்கு சூழலை அனுப்பும் ஒழுங்கமைக்கப்பட்ட குழாய்துளையில் ஒன்றிணைக்கப்படுகின்றன.


## சிறப்பு முகவர்களைக் உருவாக்குதல்


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## தொடர்ச்சியான பணிவழிக் கட்டமைப்பு

`WorkflowBuilder` உங்களுக்கு முகவர்களை ஒரு திசைவு கொண்ட வரைபடமாக இணைக்க அனுமதிக்கிறது. இங்கே நாம் ஒரு எளிய இரண்டு படி குழாய் உருவாக்குகிறோம்: **TravelPlanner** பயணத் திட்டத்தை வரைபடம் வரைபடுகிறது, பிறகு **TravelConcierge** அதை ஆய்வு செய்து மேம்படுத்துகிறது.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## வேலைப்பாட்டில் மேலும் முகவர்களைச் சேர்்்தல்

பன்முகவர் வடிவத்தின் மிக பெரிய நன்மைகளுள் ஒன்று அதை விரிவுபடுத்த எளிதாக இருந்தே இருக்கும் என்பது. கீழே நாம் பயணியின் பட்ஜெட்டுக்கு எதிராக திட்டத்தை சரிபார்க்கும், செலவுகளை எல்லைக்கு மிச்சமாக்கும் பொருட்களை குறிக்கின்ற, மற்றும் பணச் சேமிப்பான மாற்றுகளை பரிந்துரைக்கும் **BudgetReviewer** முகவரைக் கூடியுள்ளோம். இப்போது வேலைப்பாடு பயணிகள் நிச்சயமாக அமைவதாக மூன்று முகவர்களை தூண்டுகிறது:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எப்படி செய்வது என்பதை கற்றுக்கொண்டீர்கள்:

1. **சிறப்பான முகவர்கள் உருவாக்குதல்** — ஒவ்வொரு முகவருக்கும் கவனம் செலுத்திய பங்கு (திட்டமிடல், தனிப்பயன் சேவை, பட்ஜெட் மதிப்பாய்வு).
2. **`WorkflowBuilder` மற்றும் `add_edge` பயன்படுத்தி முகவர்களை தொடர்ச்சி செயல்முறை ஒன்றாக இணைத்தல்**.
3. **பல முகவர் குழாய் வழியே வெளியீட்டை நேரடியாக வழங்குதல்**, எது முகவர் பேசுகிறான் என்பதைக் கண்காணித்தல்.
4. **தற்போதையவற்றை மாற்றாமல் புதிய முகவர்களை சங்கிலியில் சேர்்தல்** மூலம் ஒரு பணிச்சூழலை விரிவுபடுத்துதல்.

பல முகவர் வடிவமைப்பு ஒவ்வொரு முகவரையும் எளிமையாக வைத்திருக்கவும், ஒற்றை முகவர் ஒரே நேரத்தில் பெற முடியாத சிறந்த, மேலாண்மை செய்யப்பட்ட முடிவுகளை வழங்கவும் உதவுகிறது.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**மறுப்பு**:
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்திற்காக முயற்சி செய்துள்ளோம், ஆனால் தானாக செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். அசல் ஆவணம் அதன் தாய்மொழியில் அதிகாரப்பூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்நுட்பமான மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் ஏற்படும் எந்த தவறான புரிதல்கள் அல்லது தவறான விளக்கத்திற்கும் நாங்கள் பொறுப்பில்வில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
